### Common hardware interface and closed-loop order

The output adapter consumes:

- `pulse_waveform_normalized`
- `PULSE_REPETITION_FREQUENCY_HZ`
- `PULSE_LOW_VOLTAGE_V`
- `PULSE_HIGH_VOLTAGE_V`

The Red Pitaya measurement adapter acquires one waveform at a time.
For every acquired waveform, the notebook performs:

1. detect every valid pulse in the analysis window;
2. calculate every individual pulse area;
3. average the valid pulse areas to obtain one group measurement;
4. calculate a PI correction from the group-mean area;
5. apply the correction to the Red Pitaya DAC before the next group.

The correction convention is:

`area_error = measured_group_mean_area - target_group_mean_area`

A positive error therefore means that the measured pulse group is too large.


In [ ]:
import csv
import math
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# 1. Pulse Making

In [ ]:
# 1.1 Common pulse definition

PULSE_REPETITION_FREQUENCY_HZ = 5000.0

PULSE_LOW_VOLTAGE_V = 0.0
PULSE_HIGH_VOLTAGE_V = 1.0

# Each entry is:
# (normalized level, fraction of one period)
PULSE_SEQUENCE = [
    (1.0, 0.05),
    (0.0, 0.1),
    (1.0, 0.10),
    (0.0, 0.75),
]

PULSE_WAVEFORM_POINTS = 16384


# Parameter checks
fraction_sum = sum(
    fraction
    for _, fraction in PULSE_SEQUENCE
)

if not math.isclose(
    fraction_sum,
    1.0,
    abs_tol=1e-9,
):
    raise ValueError(
        f"Pulse fractions sum to {fraction_sum}, not 1.0."
    )

if PULSE_HIGH_VOLTAGE_V <= PULSE_LOW_VOLTAGE_V:
    raise ValueError(
        "PULSE_HIGH_VOLTAGE_V must exceed "
        "PULSE_LOW_VOLTAGE_V."
    )

for level, fraction in PULSE_SEQUENCE:
    if not 0.0 <= level <= 1.0:
        raise ValueError(
            "Pulse level must lie between 0 and 1."
        )

    if fraction <= 0.0:
        raise ValueError(
            "Pulse fraction must be positive."
        )


# Construct exactly one normalized waveform period.
pulse_waveform_normalized = np.empty(
    PULSE_WAVEFORM_POINTS,
    dtype=np.float32,
)

start_index = 0
cumulative_fraction = 0.0

for sequence_index, (level, fraction) in enumerate(
    PULSE_SEQUENCE
):
    cumulative_fraction += fraction

    if sequence_index == len(PULSE_SEQUENCE) - 1:
        end_index = PULSE_WAVEFORM_POINTS
    else:
        end_index = int(
            round(
                cumulative_fraction
                * PULSE_WAVEFORM_POINTS
            )
        )

    pulse_waveform_normalized[
        start_index:end_index
    ] = level

    start_index = end_index


pulse_period_s = (
    1.0
    / PULSE_REPETITION_FREQUENCY_HZ
)


print(
    "Period:",
    pulse_period_s * 1000000.0,
    "us",
)



for level, fraction in PULSE_SEQUENCE:
    voltage_v = (
        PULSE_LOW_VOLTAGE_V
        + level
        * (
            PULSE_HIGH_VOLTAGE_V
            - PULSE_LOW_VOLTAGE_V
        )
    )

    duration_us = (
        fraction
        * pulse_period_s
        * 1000000.0
    )

    print(
        f"level={level:.3f}, "
        f"voltage={voltage_v:.6f} V, "
        f"duration={duration_us:.6f} us"
    )


## 1.2 For Red Pitaya: connection and pulse output


In [ ]:
# 1.2 For Red Pitaya — SCPI connection and pulse output

import redpitaya_scpi as scpi


REDPITAYA_HOST = "192.168.50.2"
REDPITAYA_OUTPUT_CHANNEL = 1
REDPITAYA_COMMAND_DELAY_S = 0.05


try:
    rp.close()
except Exception:
    pass


rp = scpi.scpi(
    REDPITAYA_HOST
)

print(
    "Connected:",
    rp.txrx_txt("*IDN?")
)


def send_redpitaya(
    command,
    delay_s=REDPITAYA_COMMAND_DELAY_S,
):
    payload = (
        command + "\r\n"
    ).encode("utf-8")

    # sendall ensures that long arbitrary-waveform
    # commands are transmitted completely.
    rp._socket.sendall(payload)

    time.sleep(delay_s)


def query_redpitaya(command):
    return rp.txrx_txt(command)


def configure_redpitaya_generator():
    channel = REDPITAYA_OUTPUT_CHANNEL

    # Red Pitaya arbitrary-waveform memory uses values
    # between -1 and +1.
    redpitaya_waveform = (
        2.0
        * pulse_waveform_normalized
        - 1.0
    )

    # Important: no opening or closing braces around
    # the comma-separated waveform values.
    waveform_text = ",".join(
        f"{float(value):.7g}"
        for value in redpitaya_waveform
    )

    upload_command = (
        "SOUR"
        + str(channel)
        + ":TRAC:DATA:DATA "
        + waveform_text
    )

    amplitude_v = (
        PULSE_HIGH_VOLTAGE_V
        - PULSE_LOW_VOLTAGE_V
    ) / 2.0

    offset_v = (
        PULSE_HIGH_VOLTAGE_V
        + PULSE_LOW_VOLTAGE_V
    ) / 2.0

    send_redpitaya(
        "GEN:RST"
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":FUNC ARBITRARY"
    )

    send_redpitaya(
        upload_command,
        delay_s=1.0,
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":FREQ:FIX "
        + str(PULSE_REPETITION_FREQUENCY_HZ)
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":VOLT "
        + str(amplitude_v)
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":VOLT:OFFS "
        + str(offset_v)
    )

    send_redpitaya(
        "OUTPUT"
        + str(channel)
        + ":STATE ON"
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":TRIG:INT"
    )

    print(
        "Pulse output started on OUT"
        + str(channel)
        + "."
    )


configure_redpitaya_generator()


# 2. Parameters for Pulse Measurements

In [ ]:
# 2.1 Common parameters for pulse measurements

# Exact duration analysed inside every acquired waveform.
# The Red Pitaya acquisition buffer can be slightly longer.
TIME_WINDOW_TO_COVER_S = 0.000005

# Duration of the complete experiment.
TOTAL_MEASUREMENT_TIME_S = 10

# Time between the scheduled starts of neighbouring captures.
MEASUREMENT_INTERVAL_S = 1

# Software pulse-detection threshold:
# local baseline + THRESHOLD_FRACTION
#                * TARGET_PULSE_HEIGHT_ABOVE_BASELINE_V
TARGET_PULSE_HEIGHT_ABOVE_BASELINE_V = 0.10
THRESHOLD_FRACTION = 0.3
CONSECUTIVE_TRIGGER_SAMPLES = 3

# Pulse-specific baseline windows and guard regions,
# all defined in sample counts.
PRE_BASELINE_SAMPLES = 32
PRE_BASELINE_GUARD_SAMPLES = 32

POST_BASELINE_SAMPLES = 32
POST_BASELINE_GUARD_SAMPLES = 32

# Pulse-duration validity checks, also in sample counts.
MIN_VALID_PULSE_SAMPLES = 3
MAX_VALID_PULSE_SAMPLES = 8000

# Safety limit protecting against an accidental infinite
# pulse-detection loop.
MAX_PULSES_PER_CAPTURE = 1000

# Enter a calibrated target pulse area in V*s.
# Use None to measure pulse area without calculating an error.
TARGET_PULSE_AREA_VS = None

# Print the complete pulse-area list for every capture.
PRINT_ALL_PULSE_AREAS = True


# Saving
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIRECTORY = Path("pulse_measurement_runs") / RUN_ID
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

RAW_WAVEFORMS_PATH = OUTPUT_DIRECTORY / "raw_waveforms.npy"
CAPTURE_METADATA_PATH = OUTPUT_DIRECTORY / "capture_metadata.csv"
CAPTURE_ERRORS_PATH = OUTPUT_DIRECTORY / "capture_errors.csv"
PULSE_RESULTS_PATH = OUTPUT_DIRECTORY / "pulse_area_results.csv"
CAPTURE_PULSE_SUMMARY_PATH = (
    OUTPUT_DIRECTORY / "capture_pulse_summary.csv"
)


# Common parameter checks
if TIME_WINDOW_TO_COVER_S <= 0.0:
    raise ValueError(
        "TIME_WINDOW_TO_COVER_S must be positive."
    )

if TOTAL_MEASUREMENT_TIME_S <= 0.0:
    raise ValueError(
        "TOTAL_MEASUREMENT_TIME_S must be positive."
    )

if MEASUREMENT_INTERVAL_S <= 0.0:
    raise ValueError(
        "MEASUREMENT_INTERVAL_S must be positive."
    )

if TARGET_PULSE_HEIGHT_ABOVE_BASELINE_V <= 0.0:
    raise ValueError(
        "TARGET_PULSE_HEIGHT_ABOVE_BASELINE_V must be positive."
    )

if not 0.0 < THRESHOLD_FRACTION < 1.0:
    raise ValueError(
        "THRESHOLD_FRACTION must lie between 0 and 1."
    )

if CONSECUTIVE_TRIGGER_SAMPLES < 1:
    raise ValueError(
        "CONSECUTIVE_TRIGGER_SAMPLES must be at least 1."
    )

if PRE_BASELINE_SAMPLES < 1:
    raise ValueError(
        "PRE_BASELINE_SAMPLES must be at least 1."
    )

if POST_BASELINE_SAMPLES < 1:
    raise ValueError(
        "POST_BASELINE_SAMPLES must be at least 1."
    )

if PRE_BASELINE_GUARD_SAMPLES < 0:
    raise ValueError(
        "PRE_BASELINE_GUARD_SAMPLES cannot be negative."
    )

if POST_BASELINE_GUARD_SAMPLES < 0:
    raise ValueError(
        "POST_BASELINE_GUARD_SAMPLES cannot be negative."
    )

if MIN_VALID_PULSE_SAMPLES < 1:
    raise ValueError(
        "MIN_VALID_PULSE_SAMPLES must be at least 1."
    )

if MAX_VALID_PULSE_SAMPLES <= MIN_VALID_PULSE_SAMPLES:
    raise ValueError(
        "MAX_VALID_PULSE_SAMPLES must exceed "
        "MIN_VALID_PULSE_SAMPLES."
    )

if MAX_PULSES_PER_CAPTURE < 1:
    raise ValueError(
        "MAX_PULSES_PER_CAPTURE must be at least 1."
    )

if (
    TARGET_PULSE_AREA_VS is not None
    and TARGET_PULSE_AREA_VS <= 0.0
):
    raise ValueError(
        "TARGET_PULSE_AREA_VS must be positive or None."
    )

print("Output folder:", OUTPUT_DIRECTORY.resolve())


## 2.2 For Red Pitaya — acquisition parameters

This subsection converts the requested time window into a supported Red Pitaya decimation and supplies `SAMPLE_INTERVAL_S` and `TRIGGER_INDEX` to the common analysis.

In [ ]:
# 2.2 For Red Pitaya — acquisition parameters and sampling rate

# Red Pitaya STEMlab 125-14 input settings
REDPITAYA_INPUT_CHANNEL = 1
REDPITAYA_INPUT_GAIN = "LV"

# Coarse hardware trigger in volts.
REDPITAYA_HARDWARE_TRIGGER_LEVEL_V = -0.05

REDPITAYA_ADC_CLOCK_HZ = 125000000.0
REDPITAYA_ADC_BUFFER_SIZE = 16384

# Decimations used by this Red Pitaya SCPI environment.
REDPITAYA_SUPPORTED_DECIMATIONS = np.array(
    [
        1,
        8,
        16,
        32,
        64,
        128,
        256,
        512,
        1024,
        2048,
        4096,
        8192,
        16384,
        32768,
        65536,
    ],
    dtype=int,
)

REDPITAYA_TRIGGER_TIMEOUT_S = 5.0
REDPITAYA_SOCKET_TIMEOUT_S = 10.0
REDPITAYA_MAX_RETRIES_PER_CAPTURE = 2


# Select the smallest available decimation whose full
# 16384-point acquisition buffer covers the requested window.
ideal_decimation = (
    TIME_WINDOW_TO_COVER_S
    * REDPITAYA_ADC_CLOCK_HZ
    / REDPITAYA_ADC_BUFFER_SIZE
)

available_decimations = REDPITAYA_SUPPORTED_DECIMATIONS[
    REDPITAYA_SUPPORTED_DECIMATIONS
    >= ideal_decimation
]

if available_decimations.size == 0:
    raise ValueError(
        "The requested time window is longer than the maximum "
        "Red Pitaya acquisition window."
    )

REDPITAYA_DECIMATION = int(
    available_decimations[0]
)

ACTUAL_SAMPLING_RATE_HZ = (
    REDPITAYA_ADC_CLOCK_HZ
    / REDPITAYA_DECIMATION
)

SAMPLE_INTERVAL_S = (
    1.0
    / ACTUAL_SAMPLING_RATE_HZ
)

ACTUAL_TIME_WINDOW_S = (
    REDPITAYA_ADC_BUFFER_SIZE
    * SAMPLE_INTERVAL_S
)


# Number of samples corresponding to the fixed common
# analysis window. The nearest representable duration is used.
ANALYSIS_WINDOW_SAMPLES = int(
    round(
        TIME_WINDOW_TO_COVER_S
        / SAMPLE_INTERVAL_S
    )
)

ANALYSIS_WINDOW_ACTUAL_S = (
    ANALYSIS_WINDOW_SAMPLES
    * SAMPLE_INTERVAL_S
)

if ANALYSIS_WINDOW_SAMPLES < 1:
    raise ValueError(
        "The requested analysis window contains no samples."
    )

if (
    ANALYSIS_WINDOW_SAMPLES
    > REDPITAYA_ADC_BUFFER_SIZE
):
    raise ValueError(
        "The requested analysis window does not fit "
        "inside one Red Pitaya acquisition buffer."
    )


# Keep some samples before the hardware trigger so that
# the first triggered pulse still has a complete pre-baseline.
ANALYSIS_PRE_TRIGGER_SAMPLES = max(
    128,
    PRE_BASELINE_SAMPLES
    + PRE_BASELINE_GUARD_SAMPLES
    + CONSECUTIVE_TRIGGER_SAMPLES
    + 16,
)

# Place the hardware trigger near the beginning of the
# returned buffer, while retaining extra safety samples.
TRIGGER_INDEX = max(
    256,
    ANALYSIS_PRE_TRIGGER_SAMPLES + 64,
)

# ACQ:TRIG:DLY 0 puts the trigger at sample 8192.
# A positive delay moves it towards the beginning.
REDPITAYA_TRIGGER_DELAY_SAMPLES = (
    REDPITAYA_ADC_BUFFER_SIZE // 2
    - TRIGGER_INDEX
)

if REDPITAYA_TRIGGER_DELAY_SAMPLES < 0:
    raise ValueError(
        "The selected trigger index would require a "
        "negative Red Pitaya trigger delay."
    )


# Analyse exactly TIME_WINDOW_TO_COVER_S around the
# trigger, beginning slightly before the trigger so the
# first pulse and its baseline are complete.
ANALYSIS_WINDOW_START_INDEX = (
    TRIGGER_INDEX
    - ANALYSIS_PRE_TRIGGER_SAMPLES
)

ANALYSIS_WINDOW_END_INDEX = (
    ANALYSIS_WINDOW_START_INDEX
    + ANALYSIS_WINDOW_SAMPLES
)

if ANALYSIS_WINDOW_START_INDEX < 0:
    raise ValueError(
        "The analysis window starts before the buffer."
    )

if (
    ANALYSIS_WINDOW_END_INDEX
    > REDPITAYA_ADC_BUFFER_SIZE
):
    raise ValueError(
        "The complete analysis window does not fit after "
        "the selected trigger position."
    )


MAX_CAPTURE_ATTEMPTS = (
    int(
        math.floor(
            TOTAL_MEASUREMENT_TIME_S
            / MEASUREMENT_INTERVAL_S
        )
    )
    + 1
)


# Red Pitaya-specific checks
if REDPITAYA_INPUT_CHANNEL not in (1, 2):
    raise ValueError(
        "REDPITAYA_INPUT_CHANNEL must be 1 or 2."
    )

if REDPITAYA_INPUT_GAIN not in ("LV", "HV"):
    raise ValueError(
        'REDPITAYA_INPUT_GAIN must be "LV" or "HV".'
    )


print("Red Pitaya acquisition parameters ready.")
print("Requested analysis window:", TIME_WINDOW_TO_COVER_S, "s")
print("Represented analysis window:", ANALYSIS_WINDOW_ACTUAL_S, "s")
print("Selected decimation:", REDPITAYA_DECIMATION)
print("Actual sampling rate:", ACTUAL_SAMPLING_RATE_HZ, "samples/s")
print("Sample interval:", SAMPLE_INTERVAL_S, "s")
print("Full acquired buffer:", ACTUAL_TIME_WINDOW_S, "s")
print("Trigger index:", TRIGGER_INDEX)
print(
    "Red Pitaya trigger delay:",
    REDPITAYA_TRIGGER_DELAY_SAMPLES,
    "samples",
)
print(
    "Analysis sample range:",
    ANALYSIS_WINDOW_START_INDEX,
    "to",
    ANALYSIS_WINDOW_END_INDEX - 1,
)


# 3. Pulse Measuring

Section 3 defines acquisition and multi-pulse analysis functions.
The repeated measure → analyse → correct loop is executed in Section 4.2.


## 3.1 For Red Pitaya — acquire one complete waveform

This subsection only defines the Red Pitaya acquisition functions.
Running it does not start the long experiment.


In [ ]:
# 3.1 For Red Pitaya — acquire complete waveforms

def parse_redpitaya_scpi_array(raw_text):
    text = raw_text.strip()

    if text.startswith("{") and text.endswith("}"):
        text = text[1:-1]

    if not text:
        raise ValueError(
            "Red Pitaya returned an empty waveform."
        )

    waveform = np.fromstring(
        text,
        sep=",",
        dtype=np.float64,
    )

    if waveform.size != REDPITAYA_ADC_BUFFER_SIZE:
        raise ValueError(
            "Expected "
            + str(REDPITAYA_ADC_BUFFER_SIZE)
            + " samples, received "
            + str(waveform.size)
            + "."
        )

    return waveform.astype(np.float32)


def configure_redpitaya_acquisition():
    send_redpitaya("ACQ:RST")

    send_redpitaya(
        "ACQ:DEC "
        + str(REDPITAYA_DECIMATION)
    )

    send_redpitaya(
        "ACQ:SOUR"
        + str(REDPITAYA_INPUT_CHANNEL)
        + ":GAIN "
        + REDPITAYA_INPUT_GAIN
    )

    send_redpitaya("ACQ:DATA:FORMAT ASCII")
    send_redpitaya("ACQ:DATA:UNITS VOLTS")

    send_redpitaya(
        "ACQ:TRIG:LEV "
        + str(REDPITAYA_HARDWARE_TRIGGER_LEVEL_V)
    )

    send_redpitaya(
        "ACQ:TRIG:DLY "
        + str(REDPITAYA_TRIGGER_DELAY_SAMPLES)
    )


def wait_for_redpitaya_trigger():
    deadline_s = (
        time.monotonic()
        + REDPITAYA_TRIGGER_TIMEOUT_S
    )

    while time.monotonic() < deadline_s:
        trigger_status = query_redpitaya(
            "ACQ:TRIG:STAT?"
        ).strip().upper()

        if trigger_status.startswith("TD"):
            return time.monotonic()

        time.sleep(0.001)

    raise TimeoutError(
        "No Red Pitaya hardware trigger arrived within "
        + str(REDPITAYA_TRIGGER_TIMEOUT_S)
        + " s."
    )


def wait_for_redpitaya_buffer_fill():
    # ACQ:TRIG:FILL? is unavailable on this old
    # INSTR2020 / ecosystem 01-02 SCPI server.
    #
    # Because the trigger is now near the beginning of
    # the buffer, wait for every required post-trigger
    # sample before requesting the waveform.
    post_trigger_samples = (
        REDPITAYA_ADC_BUFFER_SIZE
        - TRIGGER_INDEX
    )

    post_trigger_wait_s = max(
        0.001,
        1.10
        * post_trigger_samples
        * SAMPLE_INTERVAL_S,
    )

    time.sleep(post_trigger_wait_s)


def acquire_one_redpitaya_waveform():
    configure_redpitaya_acquisition()

    send_redpitaya("ACQ:START")

    # Fill enough fresh circular-buffer data to provide
    # the requested pre-trigger baseline.
    prefill_wait_s = max(
        0.001,
        1.10
        * TRIGGER_INDEX
        * SAMPLE_INTERVAL_S,
    )

    time.sleep(prefill_wait_s)

    send_redpitaya(
        "ACQ:TRIG CH"
        + str(REDPITAYA_INPUT_CHANNEL)
        + "_PE"
    )

    trigger_time_s = wait_for_redpitaya_trigger()
    wait_for_redpitaya_buffer_fill()

    raw_text = query_redpitaya(
        "ACQ:SOUR"
        + str(REDPITAYA_INPUT_CHANNEL)
        + ":DATA?"
    )

    waveform_v = parse_redpitaya_scpi_array(
        raw_text
    )

    return waveform_v, trigger_time_s



## 3.2 Common multi-pulse area algorithm

The algorithm detects all valid pulses in one acquired waveform.

For each pulse:

`pulse_area = sum(sample - baseline) * sample_interval`

`pulse_height_v` is the mean baseline-subtracted height in the detected
pulse window. `peak_height_v` is stored separately as the largest height.


In [ ]:
# 3.2 Common multi-pulse area algorithm

def first_consecutive_true(mask, required_count):
    mask = np.asarray(
        mask,
        dtype=np.int8,
    )

    if mask.size < required_count:
        return None

    running_count = np.convolve(
        mask,
        np.ones(
            required_count,
            dtype=np.int16,
        ),
        mode="valid",
    )

    matching_indices = np.flatnonzero(
        running_count == required_count
    )

    if matching_indices.size == 0:
        return None

    return int(
        matching_indices[0]
    )


def make_empty_pulse_result(
    pulse_index_in_capture,
    reason,
):
    return {
        "pulse_index_in_capture": pulse_index_in_capture,
        "valid": False,
        "reason": reason,
        "pulse_area_vs": np.nan,
        "pulse_area_v_us": np.nan,
        "baseline_pre_v": np.nan,
        "baseline_post_v": np.nan,
        "baseline_for_area_v": np.nan,
        "threshold_v": np.nan,
        "rising_index": np.nan,
        "falling_index": np.nan,
        "pulse_samples": np.nan,
        "pulse_duration_s": np.nan,
        "pulse_height_v": np.nan,
        "peak_height_v": np.nan,
        "area_error_vs": np.nan,
        "normalized_area_error": np.nan,
    }


def analyse_all_pulses_in_waveform(
    samples_v,
    sample_interval_s,
    analysis_start_index,
    analysis_end_index,
    target_pulse_height_v,
    threshold_fraction,
    consecutive_trigger_samples,
    pre_baseline_samples,
    pre_baseline_guard_samples,
    post_baseline_samples,
    post_baseline_guard_samples,
    minimum_valid_pulse_samples,
    maximum_valid_pulse_samples,
    maximum_pulses_per_capture,
    target_pulse_area_vs=None,
):
    samples_v = np.asarray(
        samples_v,
        dtype=np.float64,
    )

    if (
        analysis_start_index < 0
        or analysis_end_index > samples_v.size
        or analysis_end_index <= analysis_start_index
    ):
        return [
            make_empty_pulse_result(
                pulse_index_in_capture=np.nan,
                reason="analysis_window_unavailable",
            )
        ]

    initial_baseline_start = (
        analysis_start_index
    )

    initial_baseline_end = (
        initial_baseline_start
        + pre_baseline_samples
    )

    if (
        initial_baseline_end
        > analysis_end_index
    ):
        return [
            make_empty_pulse_result(
                pulse_index_in_capture=np.nan,
                reason="initial_baseline_unavailable",
            )
        ]

    baseline_reference_v = float(
        np.mean(
            samples_v[
                initial_baseline_start:
                initial_baseline_end
            ]
        )
    )

    search_threshold_v = (
        baseline_reference_v
        + threshold_fraction
        * target_pulse_height_v
    )

    search_index = (
        analysis_start_index
        + pre_baseline_samples
        + pre_baseline_guard_samples
    )

    pulse_results = []
    pulse_index_in_capture = 0

    while (
        search_index < analysis_end_index
        and pulse_index_in_capture
        < maximum_pulses_per_capture
    ):
        rising_relative_index = first_consecutive_true(
            samples_v[
                search_index:
                analysis_end_index
            ]
            > search_threshold_v,
            consecutive_trigger_samples,
        )

        if rising_relative_index is None:
            break

        candidate_rising_index = (
            search_index
            + rising_relative_index
        )

        pre_end = (
            candidate_rising_index
            - pre_baseline_guard_samples
        )

        pre_start = (
            pre_end
            - pre_baseline_samples
        )

        if pre_start < analysis_start_index:
            search_index = (
                candidate_rising_index
                + consecutive_trigger_samples
            )
            continue

        baseline_pre_v = float(
            np.mean(
                samples_v[
                    pre_start:
                    pre_end
                ]
            )
        )

        threshold_v = (
            baseline_pre_v
            + threshold_fraction
            * target_pulse_height_v
        )

        # Refine the rising edge using the pulse-specific
        # local baseline and threshold.
        refined_search_start = max(
            search_index,
            pre_end,
        )

        refined_rising_relative_index = (
            first_consecutive_true(
                samples_v[
                    refined_search_start:
                    analysis_end_index
                ]
                > threshold_v,
                consecutive_trigger_samples,
            )
        )

        if refined_rising_relative_index is None:
            break

        rising_index = (
            refined_search_start
            + refined_rising_relative_index
        )

        # Recalculate the pre-baseline at the final edge.
        pre_end = (
            rising_index
            - pre_baseline_guard_samples
        )

        pre_start = (
            pre_end
            - pre_baseline_samples
        )

        if pre_start < analysis_start_index:
            search_index = (
                rising_index
                + consecutive_trigger_samples
            )
            continue

        baseline_pre_v = float(
            np.mean(
                samples_v[
                    pre_start:
                    pre_end
                ]
            )
        )

        threshold_v = (
            baseline_pre_v
            + threshold_fraction
            * target_pulse_height_v
        )

        falling_search_start = (
            rising_index
            + consecutive_trigger_samples
        )

        falling_relative_index = first_consecutive_true(
            samples_v[
                falling_search_start:
                analysis_end_index
            ]
            < threshold_v,
            consecutive_trigger_samples,
        )

        if falling_relative_index is None:
            result = make_empty_pulse_result(
                pulse_index_in_capture,
                "no_falling_edge",
            )

            result.update(
                {
                    "baseline_pre_v": baseline_pre_v,
                    "threshold_v": threshold_v,
                    "rising_index": rising_index,
                }
            )

            pulse_results.append(result)
            break

        first_below_index = (
            falling_search_start
            + falling_relative_index
        )

        falling_index = (
            first_below_index - 1
        )

        pulse_samples = (
            falling_index
            - rising_index
            + 1
        )

        if pulse_samples < minimum_valid_pulse_samples:
            result = make_empty_pulse_result(
                pulse_index_in_capture,
                "pulse_too_short",
            )

            result.update(
                {
                    "baseline_pre_v": baseline_pre_v,
                    "threshold_v": threshold_v,
                    "rising_index": rising_index,
                    "falling_index": falling_index,
                    "pulse_samples": pulse_samples,
                }
            )

            pulse_results.append(result)

            search_index = (
                first_below_index
                + consecutive_trigger_samples
            )

            pulse_index_in_capture += 1
            continue

        if pulse_samples > maximum_valid_pulse_samples:
            result = make_empty_pulse_result(
                pulse_index_in_capture,
                "pulse_too_long",
            )

            result.update(
                {
                    "baseline_pre_v": baseline_pre_v,
                    "threshold_v": threshold_v,
                    "rising_index": rising_index,
                    "falling_index": falling_index,
                    "pulse_samples": pulse_samples,
                }
            )

            pulse_results.append(result)

            search_index = (
                first_below_index
                + consecutive_trigger_samples
            )

            pulse_index_in_capture += 1
            continue

        post_start = (
            falling_index
            + 1
            + post_baseline_guard_samples
        )

        post_end = (
            post_start
            + post_baseline_samples
        )

        if post_end > analysis_end_index:
            result = make_empty_pulse_result(
                pulse_index_in_capture,
                "post_baseline_unavailable",
            )

            result.update(
                {
                    "baseline_pre_v": baseline_pre_v,
                    "threshold_v": threshold_v,
                    "rising_index": rising_index,
                    "falling_index": falling_index,
                    "pulse_samples": pulse_samples,
                }
            )

            pulse_results.append(result)
            break

        post_window = samples_v[
            post_start:
            post_end
        ]

        post_overlap = first_consecutive_true(
            post_window > threshold_v,
            consecutive_trigger_samples,
        )

        if post_overlap is not None:
            result = make_empty_pulse_result(
                pulse_index_in_capture,
                "pulse_overlaps_post_baseline_window",
            )

            result.update(
                {
                    "baseline_pre_v": baseline_pre_v,
                    "threshold_v": threshold_v,
                    "rising_index": rising_index,
                    "falling_index": falling_index,
                    "pulse_samples": pulse_samples,
                }
            )

            pulse_results.append(result)

            search_index = (
                falling_index
                + consecutive_trigger_samples
            )

            pulse_index_in_capture += 1
            continue

        baseline_post_v = float(
            np.mean(
                post_window
            )
        )

        baseline_for_area_v = (
            baseline_pre_v
            + baseline_post_v
        ) / 2.0

        pulse_segment_v = samples_v[
            rising_index:
            falling_index + 1
        ]

        pulse_area_vs = float(
            np.sum(
                pulse_segment_v
                - baseline_for_area_v
            )
            * sample_interval_s
        )

        pulse_height_v = float(
            np.mean(pulse_segment_v)
            - baseline_for_area_v
        )

        peak_height_v = float(
            np.max(pulse_segment_v)
            - baseline_for_area_v
        )

        result = {
            "pulse_index_in_capture": pulse_index_in_capture,
            "valid": True,
            "reason": "none",
            "pulse_area_vs": pulse_area_vs,
            "pulse_area_v_us": (
                pulse_area_vs
                * 1000000.0
            ),
            "baseline_pre_v": baseline_pre_v,
            "baseline_post_v": baseline_post_v,
            "baseline_for_area_v": baseline_for_area_v,
            "threshold_v": threshold_v,
            "rising_index": rising_index,
            "falling_index": falling_index,
            "pulse_samples": pulse_samples,
            "pulse_duration_s": (
                pulse_samples
                * sample_interval_s
            ),
            "pulse_height_v": pulse_height_v,
            "peak_height_v": peak_height_v,
            "area_error_vs": np.nan,
            "normalized_area_error": np.nan,
        }

        if target_pulse_area_vs is not None:
            area_error_vs = (
                pulse_area_vs
                - target_pulse_area_vs
            )

            result["area_error_vs"] = (
                area_error_vs
            )

            result["normalized_area_error"] = (
                area_error_vs
                / target_pulse_area_vs
            )

        pulse_results.append(result)

        # The next search begins after the pulse-specific
        # post-baseline window.
        search_index = post_end

        baseline_reference_v = (
            baseline_post_v
        )

        search_threshold_v = (
            baseline_reference_v
            + threshold_fraction
            * target_pulse_height_v
        )

        pulse_index_in_capture += 1

    if not pulse_results:
        pulse_results.append(
            make_empty_pulse_result(
                pulse_index_in_capture=np.nan,
                reason="no_pulse_detected",
            )
        )

    return pulse_results


## 3.3 Common — measure and analyse one pulse group

A pulse group is all valid pulses detected in one acquired analysis window.

The group measurement passed to the controller is:

`mean_group_area = sum(valid pulse areas) / number of valid pulses`


In [ ]:
# 3.3 Common — analyse one acquired pulse group

def analyse_one_pulse_group(
    capture_index,
    waveform_v,
    trigger_time_s,
    experiment_start_s,
):
    capture_trigger_time_s = (
        trigger_time_s
        - experiment_start_s
    )

    pulse_results_for_group = (
        analyse_all_pulses_in_waveform(
            samples_v=waveform_v,
            sample_interval_s=SAMPLE_INTERVAL_S,
            analysis_start_index=(
                ANALYSIS_WINDOW_START_INDEX
            ),
            analysis_end_index=(
                ANALYSIS_WINDOW_END_INDEX
            ),
            target_pulse_height_v=(
                TARGET_PULSE_HEIGHT_ABOVE_BASELINE_V
            ),
            threshold_fraction=(
                THRESHOLD_FRACTION
            ),
            consecutive_trigger_samples=(
                CONSECUTIVE_TRIGGER_SAMPLES
            ),
            pre_baseline_samples=(
                PRE_BASELINE_SAMPLES
            ),
            pre_baseline_guard_samples=(
                PRE_BASELINE_GUARD_SAMPLES
            ),
            post_baseline_samples=(
                POST_BASELINE_SAMPLES
            ),
            post_baseline_guard_samples=(
                POST_BASELINE_GUARD_SAMPLES
            ),
            minimum_valid_pulse_samples=(
                MIN_VALID_PULSE_SAMPLES
            ),
            maximum_valid_pulse_samples=(
                MAX_VALID_PULSE_SAMPLES
            ),
            maximum_pulses_per_capture=(
                MAX_PULSES_PER_CAPTURE
            ),
            target_pulse_area_vs=(
                TARGET_PULSE_AREA_VS
            ),
        )
    )

    analysis_window_start_time_s = (
        capture_trigger_time_s
        + (
            ANALYSIS_WINDOW_START_INDEX
            - TRIGGER_INDEX
        )
        * SAMPLE_INTERVAL_S
    )

    pulse_rows = []
    valid_areas_v_us = []
    valid_heights_v = []

    for pulse_result in pulse_results_for_group:
        rising_index = pulse_result[
            "rising_index"
        ]

        if np.isfinite(rising_index):
            pulse_time_from_trigger_s = (
                rising_index
                - TRIGGER_INDEX
            ) * SAMPLE_INTERVAL_S

            pulse_time_in_analysis_window_s = (
                rising_index
                - ANALYSIS_WINDOW_START_INDEX
            ) * SAMPLE_INTERVAL_S

            pulse_measurement_time_s = (
                capture_trigger_time_s
                + pulse_time_from_trigger_s
            )
        else:
            pulse_time_from_trigger_s = np.nan
            pulse_time_in_analysis_window_s = np.nan
            pulse_measurement_time_s = np.nan

        pulse_result.update(
            {
                "capture_index": capture_index,
                "capture_trigger_time_s": (
                    capture_trigger_time_s
                ),
                "analysis_window_start_time_s": (
                    analysis_window_start_time_s
                ),
                "pulse_time_from_trigger_s": (
                    pulse_time_from_trigger_s
                ),
                "pulse_time_in_analysis_window_s": (
                    pulse_time_in_analysis_window_s
                ),
                "pulse_measurement_time_s": (
                    pulse_measurement_time_s
                ),
            }
        )

        pulse_rows.append(
            pulse_result
        )

        if pulse_result["valid"]:
            valid_areas_v_us.append(
                float(
                    pulse_result[
                        "pulse_area_v_us"
                    ]
                )
            )

            valid_heights_v.append(
                float(
                    pulse_result[
                        "pulse_height_v"
                    ]
                )
            )

    valid_pulse_count = len(
        valid_areas_v_us
    )

    if valid_pulse_count > 0:
        sum_pulse_area_v_us = float(
            np.sum(
                valid_areas_v_us
            )
        )

        mean_pulse_area_v_us = (
            sum_pulse_area_v_us
            / valid_pulse_count
        )

        pulse_area_std_v_us = float(
            np.std(
                valid_areas_v_us,
                ddof=0,
            )
        )

        mean_pulse_height_v = float(
            np.mean(
                valid_heights_v
            )
        )
    else:
        sum_pulse_area_v_us = np.nan
        mean_pulse_area_v_us = np.nan
        pulse_area_std_v_us = np.nan
        mean_pulse_height_v = np.nan

    group_summary = {
        "capture_index": capture_index,
        "capture_trigger_time_s": (
            capture_trigger_time_s
        ),
        "analysis_window_start_time_s": (
            analysis_window_start_time_s
        ),
        "valid_pulse_count": (
            valid_pulse_count
        ),
        "total_candidate_count": (
            len(
                pulse_results_for_group
            )
        ),
        "sum_pulse_area_v_us": (
            sum_pulse_area_v_us
        ),
        "mean_pulse_area_v_us": (
            mean_pulse_area_v_us
        ),
        "pulse_area_std_v_us": (
            pulse_area_std_v_us
        ),
        "mean_pulse_height_v": (
            mean_pulse_height_v
        ),
        "pulse_areas_v_us": (
            repr(
                valid_areas_v_us
            )
        ),
    }

    return pulse_rows, group_summary


# 4. Correction

Each correction iteration follows:

`acquire one waveform → calculate all pulse areas → average them → PI update → change DAC output`


## 4.1 PI parameters and area-to-DAC conversion

The error convention is:

`error = measured group-mean area - target group-mean area`

The PI controller first produces an equivalent correction in pulse-area
units. `AREA_TO_DAC_GAIN_V_PER_V_US` then converts that quantity into
Red Pitaya DAC volts.

The conversion factor is kept separate from `KP` and `KI_PER_S`, so the
physical area-to-amplitude calibration is explicit rather than hidden
inside the control gains.


In [ ]:
# 4.1 Common PI correction parameters

FEEDBACK_ENABLED = True

# Absolute target for the mean area of all valid pulses
# in one acquired group, in V*us.
#
# None + USE_FIRST_VALID_GROUP_AS_TARGET=True means the
# first valid group becomes the setpoint.
TARGET_MEAN_PULSE_AREA_V_US = None
USE_FIRST_VALID_GROUP_AS_TARGET = True

# Skip correction if too few valid pulses were detected.
MIN_VALID_PULSES_PER_GROUP = 1

# Deadband can be specified absolutely or as a fraction
# of the active target. None uses the fractional value.
AREA_ERROR_DEADBAND_V_US = None
AREA_ERROR_DEADBAND_FRACTION = 0.005

# Conservative initial gains. Tune experimentally.
KP = 0.20
KI_PER_S = 0.05

# DAC volts per measured V*us.
#
# None estimates:
# initial DAC span / target mean pulse area.
#
# Replace this with a measured calibration slope when
# the area-versus-DAC response has been characterized.
AREA_TO_DAC_GAIN_V_PER_V_US = None

# With error = measured - target:
# -1 means measured too high -> reduce DAC output.
DAC_CORRECTION_SIGN = -1.0

# Safety limits.
MAX_DAC_STEP_PER_UPDATE_V = 0.020
DAC_HIGH_VOLTAGE_MIN_V = (
    PULSE_LOW_VOLTAGE_V + 0.010
)
DAC_HIGH_VOLTAGE_MAX_V = 1.000

# Limit the integral contribution in area units.
MAX_INTEGRAL_AREA_TERM_FRACTION = 0.25

# Prevent a long skipped interval from causing one very
# large integral update.
MAX_CONTROLLER_DT_S = max(
    2.0 * MEASUREMENT_INTERVAL_S,
    0.001,
)

SETTLE_TIME_AFTER_CORRECTION_S = 0.020

CONTROL_HISTORY_PATH = (
    OUTPUT_DIRECTORY / "control_history.csv"
)


if TARGET_MEAN_PULSE_AREA_V_US is not None:
    if TARGET_MEAN_PULSE_AREA_V_US <= 0.0:
        raise ValueError(
            "TARGET_MEAN_PULSE_AREA_V_US must be "
            "positive or None."
        )

if (
    TARGET_MEAN_PULSE_AREA_V_US is None
    and not USE_FIRST_VALID_GROUP_AS_TARGET
):
    raise ValueError(
        "Provide TARGET_MEAN_PULSE_AREA_V_US or enable "
        "USE_FIRST_VALID_GROUP_AS_TARGET."
    )

if MIN_VALID_PULSES_PER_GROUP < 1:
    raise ValueError(
        "MIN_VALID_PULSES_PER_GROUP must be at least 1."
    )

if (
    AREA_ERROR_DEADBAND_V_US is not None
    and AREA_ERROR_DEADBAND_V_US < 0.0
):
    raise ValueError(
        "AREA_ERROR_DEADBAND_V_US cannot be negative."
    )

if AREA_ERROR_DEADBAND_FRACTION < 0.0:
    raise ValueError(
        "AREA_ERROR_DEADBAND_FRACTION cannot be negative."
    )

if KP < 0.0 or KI_PER_S < 0.0:
    raise ValueError(
        "KP and KI_PER_S cannot be negative."
    )

if (
    AREA_TO_DAC_GAIN_V_PER_V_US is not None
    and AREA_TO_DAC_GAIN_V_PER_V_US <= 0.0
):
    raise ValueError(
        "AREA_TO_DAC_GAIN_V_PER_V_US must be positive "
        "or None."
    )

if DAC_CORRECTION_SIGN not in (-1.0, 1.0):
    raise ValueError(
        "DAC_CORRECTION_SIGN must be -1.0 or 1.0."
    )

if MAX_DAC_STEP_PER_UPDATE_V <= 0.0:
    raise ValueError(
        "MAX_DAC_STEP_PER_UPDATE_V must be positive."
    )

if not (
    PULSE_LOW_VOLTAGE_V
    < DAC_HIGH_VOLTAGE_MIN_V
    < DAC_HIGH_VOLTAGE_MAX_V
):
    raise ValueError(
        "DAC high-voltage limits are inconsistent."
    )

if not (
    DAC_HIGH_VOLTAGE_MIN_V
    <= PULSE_HIGH_VOLTAGE_V
    <= DAC_HIGH_VOLTAGE_MAX_V
):
    raise ValueError(
        "Initial PULSE_HIGH_VOLTAGE_V lies outside the "
        "DAC correction limits."
    )


def calculate_pi_dac_update(
    measured_mean_area_v_us,
    target_mean_area_v_us,
    current_dac_high_voltage_v,
    integral_error_v_us_s,
    elapsed_since_update_s,
    area_to_dac_gain_v_per_v_us,
):
    raw_error_v_us = (
        measured_mean_area_v_us
        - target_mean_area_v_us
    )

    if AREA_ERROR_DEADBAND_V_US is None:
        deadband_v_us = (
            AREA_ERROR_DEADBAND_FRACTION
            * target_mean_area_v_us
        )
    else:
        deadband_v_us = (
            AREA_ERROR_DEADBAND_V_US
        )

    deadband_active = (
        abs(raw_error_v_us)
        <= deadband_v_us
    )

    if deadband_active:
        return {
            "raw_area_error_v_us": raw_error_v_us,
            "control_area_error_v_us": 0.0,
            "deadband_v_us": deadband_v_us,
            "deadband_active": True,
            "p_area_term_v_us": 0.0,
            "i_area_term_v_us": 0.0,
            "integral_error_v_us_s": (
                integral_error_v_us_s
            ),
            "requested_dac_step_v": 0.0,
            "applied_dac_step_v": 0.0,
            "new_dac_high_voltage_v": (
                current_dac_high_voltage_v
            ),
            "step_limited": False,
            "output_saturated": False,
        }

    control_error_v_us = (
        raw_error_v_us
    )

    dt_s = float(
        np.clip(
            elapsed_since_update_s,
            0.0,
            MAX_CONTROLLER_DT_S,
        )
    )

    candidate_integral_state = (
        integral_error_v_us_s
        + control_error_v_us
        * dt_s
    )

    maximum_integral_area_term_v_us = (
        MAX_INTEGRAL_AREA_TERM_FRACTION
        * target_mean_area_v_us
    )

    if KI_PER_S > 0.0:
        maximum_integral_state = (
            maximum_integral_area_term_v_us
            / KI_PER_S
        )

        candidate_integral_state = float(
            np.clip(
                candidate_integral_state,
                -maximum_integral_state,
                maximum_integral_state,
            )
        )
    else:
        candidate_integral_state = 0.0

    def evaluate_output(
        selected_integral_state,
    ):
        p_area_term_v_us = (
            KP * control_error_v_us
        )

        i_area_term_v_us = (
            KI_PER_S
            * selected_integral_state
        )

        equivalent_area_correction_v_us = (
            p_area_term_v_us
            + i_area_term_v_us
        )

        requested_dac_step_v = (
            DAC_CORRECTION_SIGN
            * area_to_dac_gain_v_per_v_us
            * equivalent_area_correction_v_us
        )

        limited_dac_step_v = float(
            np.clip(
                requested_dac_step_v,
                -MAX_DAC_STEP_PER_UPDATE_V,
                MAX_DAC_STEP_PER_UPDATE_V,
            )
        )

        proposed_high_voltage_v = (
            current_dac_high_voltage_v
            + limited_dac_step_v
        )

        new_high_voltage_v = float(
            np.clip(
                proposed_high_voltage_v,
                DAC_HIGH_VOLTAGE_MIN_V,
                DAC_HIGH_VOLTAGE_MAX_V,
            )
        )

        return {
            "p_area_term_v_us": p_area_term_v_us,
            "i_area_term_v_us": i_area_term_v_us,
            "requested_dac_step_v": (
                requested_dac_step_v
            ),
            "limited_dac_step_v": (
                limited_dac_step_v
            ),
            "proposed_high_voltage_v": (
                proposed_high_voltage_v
            ),
            "new_high_voltage_v": (
                new_high_voltage_v
            ),
        }

    evaluated = evaluate_output(
        candidate_integral_state
    )

    output_saturated = not math.isclose(
        evaluated["new_high_voltage_v"],
        evaluated["proposed_high_voltage_v"],
        rel_tol=0.0,
        abs_tol=1e-15,
    )

    # Anti-windup: discard the newest integral update
    # if the output rail prevents the requested motion.
    if output_saturated:
        candidate_integral_state = (
            integral_error_v_us_s
        )

        evaluated = evaluate_output(
            candidate_integral_state
        )

        output_saturated = not math.isclose(
            evaluated["new_high_voltage_v"],
            evaluated["proposed_high_voltage_v"],
            rel_tol=0.0,
            abs_tol=1e-15,
        )

    applied_dac_step_v = (
        evaluated["new_high_voltage_v"]
        - current_dac_high_voltage_v
    )

    step_limited = not math.isclose(
        evaluated["limited_dac_step_v"],
        evaluated["requested_dac_step_v"],
        rel_tol=0.0,
        abs_tol=1e-15,
    )

    return {
        "raw_area_error_v_us": raw_error_v_us,
        "control_area_error_v_us": (
            control_error_v_us
        ),
        "deadband_v_us": deadband_v_us,
        "deadband_active": False,
        "p_area_term_v_us": (
            evaluated["p_area_term_v_us"]
        ),
        "i_area_term_v_us": (
            evaluated["i_area_term_v_us"]
        ),
        "integral_error_v_us_s": (
            candidate_integral_state
        ),
        "requested_dac_step_v": (
            evaluated["requested_dac_step_v"]
        ),
        "applied_dac_step_v": (
            applied_dac_step_v
        ),
        "new_dac_high_voltage_v": (
            evaluated["new_high_voltage_v"]
        ),
        "step_limited": step_limited,
        "output_saturated": output_saturated,
    }


print("PI correction parameters ready.")
print("KP:", KP)
print("KI_PER_S:", KI_PER_S)
print(
    "Configured area-to-DAC gain:",
    AREA_TO_DAC_GAIN_V_PER_V_US,
)


## 4.2 For Red Pitaya — apply correction to the DAC output

This cell runs the complete closed loop.

After each completed capture it immediately writes a checkpoint containing:

- raw waveforms;
- capture metadata and errors;
- every individual pulse result;
- each pulse-group mean;
- PI terms and the applied DAC correction.

Therefore, manually interrupting this cell preserves every capture that
finished before the interruption.


In [ ]:
# 4.2 For Red Pitaya — closed-loop measurement and correction

def set_redpitaya_dac_high_voltage(
    new_high_voltage_v,
):
    low_voltage_v = (
        PULSE_LOW_VOLTAGE_V
    )

    amplitude_v = (
        new_high_voltage_v
        - low_voltage_v
    ) / 2.0

    offset_v = (
        new_high_voltage_v
        + low_voltage_v
    ) / 2.0

    channel = (
        REDPITAYA_OUTPUT_CHANNEL
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":VOLT "
        + str(amplitude_v)
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":VOLT:OFFS "
        + str(offset_v)
    )

    send_redpitaya(
        "SOUR"
        + str(channel)
        + ":TRIG:INT"
    )

    time.sleep(
        SETTLE_TIME_AFTER_CORRECTION_S
    )


def save_closed_loop_checkpoint(
    captured_waveforms,
    capture_metadata_rows,
    capture_error_rows,
    pulse_analysis_rows,
    group_summary_rows,
    control_history_rows,
):
    if captured_waveforms:
        np.save(
            RAW_WAVEFORMS_PATH,
            np.stack(
                captured_waveforms,
                axis=0,
            ),
        )

    pd.DataFrame(
        capture_metadata_rows
    ).to_csv(
        CAPTURE_METADATA_PATH,
        index=False,
    )

    pd.DataFrame(
        capture_error_rows,
        columns=[
            "capture_index",
            "attempts",
            "error_type",
            "error_message",
        ],
    ).to_csv(
        CAPTURE_ERRORS_PATH,
        index=False,
    )

    pd.DataFrame(
        pulse_analysis_rows
    ).to_csv(
        PULSE_RESULTS_PATH,
        index=False,
    )

    pd.DataFrame(
        group_summary_rows
    ).to_csv(
        CAPTURE_PULSE_SUMMARY_PATH,
        index=False,
    )

    pd.DataFrame(
        control_history_rows
    ).to_csv(
        CONTROL_HISTORY_PATH,
        index=False,
    )


captured_waveforms = []
capture_metadata_rows = []
capture_error_rows = []
analysis_rows = []
capture_summary_rows = []
control_history_rows = []

experiment_start_s = time.monotonic()
experiment_end_s = (
    experiment_start_s
    + TOTAL_MEASUREMENT_TIME_S
)

current_dac_high_voltage_v = float(
    PULSE_HIGH_VOLTAGE_V
)

active_target_area_v_us = (
    TARGET_MEAN_PULSE_AREA_V_US
)

effective_area_to_dac_gain = (
    AREA_TO_DAC_GAIN_V_PER_V_US
)

integral_error_v_us_s = 0.0
last_controller_update_s = (
    experiment_start_s
)

manual_stop_requested = False


try:
    for capture_index in range(
        MAX_CAPTURE_ATTEMPTS
    ):
        scheduled_start_s = (
            experiment_start_s
            + capture_index
            * MEASUREMENT_INTERVAL_S
        )

        if scheduled_start_s > experiment_end_s:
            break

        sleep_time_s = (
            scheduled_start_s
            - time.monotonic()
        )

        if sleep_time_s > 0.0:
            time.sleep(
                sleep_time_s
            )

        capture_succeeded = False

        for attempt_index in range(
            REDPITAYA_MAX_RETRIES_PER_CAPTURE
            + 1
        ):
            try:
                waveform_v, trigger_time_s = (
                    acquire_one_redpitaya_waveform()
                )

                if not np.all(
                    np.isfinite(
                        waveform_v
                    )
                ):
                    raise RuntimeError(
                        "Waveform contains NaN or "
                        "infinite values."
                    )

                pulse_rows, group_summary = (
                    analyse_one_pulse_group(
                        capture_index=(
                            capture_index
                        ),
                        waveform_v=(
                            waveform_v
                        ),
                        trigger_time_s=(
                            trigger_time_s
                        ),
                        experiment_start_s=(
                            experiment_start_s
                        ),
                    )
                )

                captured_waveforms.append(
                    waveform_v
                )

                capture_metadata_rows.append(
                    {
                        "capture_index": (
                            capture_index
                        ),
                        "attempts": (
                            attempt_index + 1
                        ),
                        "measurement_time_s": (
                            trigger_time_s
                            - experiment_start_s
                        ),
                        "wall_clock_time": (
                            datetime.now()
                            .astimezone()
                            .isoformat(
                                timespec="milliseconds"
                            )
                        ),
                    }
                )

                analysis_rows.extend(
                    pulse_rows
                )

                valid_pulse_count = int(
                    group_summary[
                        "valid_pulse_count"
                    ]
                )

                measured_mean_area_v_us = (
                    group_summary[
                        "mean_pulse_area_v_us"
                    ]
                )

                control_status = (
                    "not_evaluated"
                )

                raw_area_error_v_us = np.nan
                control_area_error_v_us = np.nan
                deadband_v_us = np.nan
                deadband_active = False
                p_area_term_v_us = np.nan
                i_area_term_v_us = np.nan
                requested_dac_step_v = 0.0
                applied_dac_step_v = 0.0
                step_limited = False
                output_saturated = False

                dac_high_before_v = (
                    current_dac_high_voltage_v
                )

                controller_time_s = (
                    time.monotonic()
                )

                elapsed_since_update_s = (
                    controller_time_s
                    - last_controller_update_s
                )

                valid_group_for_feedback = (
                    valid_pulse_count
                    >= MIN_VALID_PULSES_PER_GROUP
                    and np.isfinite(
                        measured_mean_area_v_us
                    )
                )

                if not valid_group_for_feedback:
                    control_status = (
                        "insufficient_valid_pulses"
                    )

                elif active_target_area_v_us is None:
                    active_target_area_v_us = float(
                        measured_mean_area_v_us
                    )

                    control_status = (
                        "target_initialized"
                    )

                    integral_error_v_us_s = 0.0
                    last_controller_update_s = (
                        controller_time_s
                    )

                elif not FEEDBACK_ENABLED:
                    control_status = (
                        "feedback_disabled"
                    )

                    raw_area_error_v_us = (
                        measured_mean_area_v_us
                        - active_target_area_v_us
                    )

                else:
                    if (
                        effective_area_to_dac_gain
                        is None
                    ):
                        initial_dac_span_v = (
                            PULSE_HIGH_VOLTAGE_V
                            - PULSE_LOW_VOLTAGE_V
                        )

                        effective_area_to_dac_gain = (
                            initial_dac_span_v
                            / active_target_area_v_us
                        )

                        print(
                            "\nAuto area-to-DAC gain:",
                            effective_area_to_dac_gain,
                            "DAC V per measured V us",
                        )

                    update = (
                        calculate_pi_dac_update(
                            measured_mean_area_v_us=(
                                measured_mean_area_v_us
                            ),
                            target_mean_area_v_us=(
                                active_target_area_v_us
                            ),
                            current_dac_high_voltage_v=(
                                current_dac_high_voltage_v
                            ),
                            integral_error_v_us_s=(
                                integral_error_v_us_s
                            ),
                            elapsed_since_update_s=(
                                elapsed_since_update_s
                            ),
                            area_to_dac_gain_v_per_v_us=(
                                effective_area_to_dac_gain
                            ),
                        )
                    )

                    raw_area_error_v_us = (
                        update[
                            "raw_area_error_v_us"
                        ]
                    )

                    control_area_error_v_us = (
                        update[
                            "control_area_error_v_us"
                        ]
                    )

                    deadband_v_us = (
                        update[
                            "deadband_v_us"
                        ]
                    )

                    deadband_active = (
                        update[
                            "deadband_active"
                        ]
                    )

                    p_area_term_v_us = (
                        update[
                            "p_area_term_v_us"
                        ]
                    )

                    i_area_term_v_us = (
                        update[
                            "i_area_term_v_us"
                        ]
                    )

                    integral_error_v_us_s = (
                        update[
                            "integral_error_v_us_s"
                        ]
                    )

                    requested_dac_step_v = (
                        update[
                            "requested_dac_step_v"
                        ]
                    )

                    applied_dac_step_v = (
                        update[
                            "applied_dac_step_v"
                        ]
                    )

                    current_dac_high_voltage_v = (
                        update[
                            "new_dac_high_voltage_v"
                        ]
                    )

                    step_limited = (
                        update[
                            "step_limited"
                        ]
                    )

                    output_saturated = (
                        update[
                            "output_saturated"
                        ]
                    )

                    if deadband_active:
                        control_status = (
                            "inside_deadband"
                        )
                    else:
                        control_status = (
                            "corrected"
                        )

                    if not math.isclose(
                        applied_dac_step_v,
                        0.0,
                        rel_tol=0.0,
                        abs_tol=1e-15,
                    ):
                        set_redpitaya_dac_high_voltage(
                            current_dac_high_voltage_v
                        )

                    last_controller_update_s = (
                        controller_time_s
                    )

                # If the target was initialized in this iteration,
                # establish the automatic physical conversion now
                # so it is visible in the saved history.
                if (
                    active_target_area_v_us is not None
                    and effective_area_to_dac_gain is None
                ):
                    initial_dac_span_v = (
                        PULSE_HIGH_VOLTAGE_V
                        - PULSE_LOW_VOLTAGE_V
                    )

                    effective_area_to_dac_gain = (
                        initial_dac_span_v
                        / active_target_area_v_us
                    )

                    print(
                        "\nAuto area-to-DAC gain:",
                        effective_area_to_dac_gain,
                        "DAC V per measured V us",
                    )

                group_summary.update(
                    {
                        "target_mean_pulse_area_v_us": (
                            active_target_area_v_us
                        ),
                        "mean_area_error_v_us": (
                            raw_area_error_v_us
                        ),
                        "dac_high_before_v": (
                            dac_high_before_v
                        ),
                        "dac_high_after_v": (
                            current_dac_high_voltage_v
                        ),
                        "control_status": (
                            control_status
                        ),
                    }
                )

                capture_summary_rows.append(
                    group_summary
                )

                control_history_rows.append(
                    {
                        "capture_index": capture_index,
                        "controller_time_s": (
                            controller_time_s
                            - experiment_start_s
                        ),
                        "valid_pulse_count": (
                            valid_pulse_count
                        ),
                        "sum_pulse_area_v_us": (
                            group_summary[
                                "sum_pulse_area_v_us"
                            ]
                        ),
                        "measured_mean_pulse_area_v_us": (
                            measured_mean_area_v_us
                        ),
                        "target_mean_pulse_area_v_us": (
                            active_target_area_v_us
                        ),
                        "raw_area_error_v_us": (
                            raw_area_error_v_us
                        ),
                        "control_area_error_v_us": (
                            control_area_error_v_us
                        ),
                        "deadband_v_us": (
                            deadband_v_us
                        ),
                        "deadband_active": (
                            deadband_active
                        ),
                        "p_area_term_v_us": (
                            p_area_term_v_us
                        ),
                        "i_area_term_v_us": (
                            i_area_term_v_us
                        ),
                        "integral_error_v_us_s": (
                            integral_error_v_us_s
                        ),
                        "area_to_dac_gain_v_per_v_us": (
                            effective_area_to_dac_gain
                        ),
                        "requested_dac_step_v": (
                            requested_dac_step_v
                        ),
                        "applied_dac_step_v": (
                            applied_dac_step_v
                        ),
                        "dac_high_before_v": (
                            dac_high_before_v
                        ),
                        "dac_high_after_v": (
                            current_dac_high_voltage_v
                        ),
                        "step_limited": (
                            step_limited
                        ),
                        "output_saturated": (
                            output_saturated
                        ),
                        "control_status": (
                            control_status
                        ),
                    }
                )

                save_closed_loop_checkpoint(
                    captured_waveforms=(
                        captured_waveforms
                    ),
                    capture_metadata_rows=(
                        capture_metadata_rows
                    ),
                    capture_error_rows=(
                        capture_error_rows
                    ),
                    pulse_analysis_rows=(
                        analysis_rows
                    ),
                    group_summary_rows=(
                        capture_summary_rows
                    ),
                    control_history_rows=(
                        control_history_rows
                    ),
                )

                print(
                    "\nCapture "
                    + str(capture_index)
                    + ": pulses="
                    + str(valid_pulse_count)
                    + ", mean area="
                    + str(measured_mean_area_v_us)
                    + " V us, target="
                    + str(active_target_area_v_us)
                    + " V us, DAC high="
                    + str(current_dac_high_voltage_v)
                    + " V, status="
                    + control_status
                )

                capture_succeeded = True
                break

            except Exception as error:
                if (
                    attempt_index
                    == REDPITAYA_MAX_RETRIES_PER_CAPTURE
                ):
                    capture_error_rows.append(
                        {
                            "capture_index": (
                                capture_index
                            ),
                            "attempts": (
                                attempt_index + 1
                            ),
                            "error_type": (
                                type(error).__name__
                            ),
                            "error_message": (
                                str(error)
                            ),
                        }
                    )

                    save_closed_loop_checkpoint(
                        captured_waveforms=(
                            captured_waveforms
                        ),
                        capture_metadata_rows=(
                            capture_metadata_rows
                        ),
                        capture_error_rows=(
                            capture_error_rows
                        ),
                        pulse_analysis_rows=(
                            analysis_rows
                        ),
                        group_summary_rows=(
                            capture_summary_rows
                        ),
                        control_history_rows=(
                            control_history_rows
                        ),
                    )

                    print(
                        "\nCapture "
                        + str(capture_index)
                        + " failed: "
                        + repr(error)
                    )
                else:
                    time.sleep(0.2)

        if not capture_succeeded:
            continue

except KeyboardInterrupt:
    manual_stop_requested = True

    print(
        "\nManual stop received. Saving all completed "
        "captures before producing the final results."
    )

finally:
    save_closed_loop_checkpoint(
        captured_waveforms=(
            captured_waveforms
        ),
        capture_metadata_rows=(
            capture_metadata_rows
        ),
        capture_error_rows=(
            capture_error_rows
        ),
        pulse_analysis_rows=(
            analysis_rows
        ),
        group_summary_rows=(
            capture_summary_rows
        ),
        control_history_rows=(
            control_history_rows
        ),
    )


if not captured_waveforms:
    raise RuntimeError(
        "No completed waveform is available."
    )


waveforms_v = np.stack(
    captured_waveforms,
    axis=0,
)

capture_metadata = pd.DataFrame(
    capture_metadata_rows
)

measurement_times_s = capture_metadata[
    "measurement_time_s"
].to_numpy(dtype=float)

capture_errors = pd.DataFrame(
    capture_error_rows,
    columns=[
        "capture_index",
        "attempts",
        "error_type",
        "error_message",
    ],
)

pulse_results = pd.DataFrame(
    analysis_rows
)

capture_pulse_summary = pd.DataFrame(
    capture_summary_rows
)

control_history = pd.DataFrame(
    control_history_rows
)


valid_pulse_mask = (
    pulse_results["valid"]
    & pulse_results[
        "pulse_measurement_time_s"
    ].notna()
)

if valid_pulse_mask.any():
    first_pulse_time_s = float(
        pulse_results.loc[
            valid_pulse_mask,
            "pulse_measurement_time_s",
        ].min()
    )

    pulse_results[
        "time_from_first_pulse_s"
    ] = (
        pulse_results[
            "pulse_measurement_time_s"
        ]
        - first_pulse_time_s
    )
else:
    pulse_results[
        "time_from_first_pulse_s"
    ] = np.nan


pulse_results.to_csv(
    PULSE_RESULTS_PATH,
    index=False,
)

capture_pulse_summary.to_csv(
    CAPTURE_PULSE_SUMMARY_PATH,
    index=False,
)

control_history.to_csv(
    CONTROL_HISTORY_PATH,
    index=False,
)


if PRINT_ALL_PULSE_AREAS:
    print(
        "\nPulse areas from every completed capture:"
    )

    for summary_row in capture_summary_rows:
        print(
            "Capture "
            + str(summary_row["capture_index"])
            + ": "
            + str(summary_row["valid_pulse_count"])
            + " valid pulses; areas (V us) = "
            + summary_row["pulse_areas_v_us"]
        )


valid_results = pulse_results[
    valid_pulse_mask
].copy()

if not valid_results.empty:
    plt.figure(
        figsize=(11, 5)
    )

    plt.scatter(
        valid_results[
            "time_from_first_pulse_s"
        ],
        valid_results[
            "pulse_area_v_us"
        ],
        s=14,
        label="Every measured pulse",
    )

    if active_target_area_v_us is not None:
        plt.axhline(
            active_target_area_v_us,
            linestyle="--",
            label="Group-mean target",
        )

    plt.xlabel(
        "Time since first detected pulse (s)"
    )

    plt.ylabel(
        "Pulse area (V us)"
    )

    plt.title(
        "All individual pulse areas versus time"
    )

    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


valid_control_history = control_history[
    control_history[
        "measured_mean_pulse_area_v_us"
    ].notna()
].copy()

if not valid_control_history.empty:
    plt.figure(
        figsize=(11, 5)
    )

    plt.scatter(
        valid_control_history[
            "controller_time_s"
        ],
        valid_control_history[
            "measured_mean_pulse_area_v_us"
        ],
        s=20,
        label="Measured group mean",
    )

    if active_target_area_v_us is not None:
        plt.axhline(
            active_target_area_v_us,
            linestyle="--",
            label="Target group mean",
        )

        if AREA_ERROR_DEADBAND_V_US is None:
            plotted_deadband_v_us = (
                AREA_ERROR_DEADBAND_FRACTION
                * active_target_area_v_us
            )
        else:
            plotted_deadband_v_us = (
                AREA_ERROR_DEADBAND_V_US
            )

        plt.axhline(
            active_target_area_v_us
            + plotted_deadband_v_us,
            linestyle=":",
            label="Deadband limits",
        )

        plt.axhline(
            active_target_area_v_us
            - plotted_deadband_v_us,
            linestyle=":",
        )

    plt.xlabel(
        "Time since experiment start (s)"
    )

    plt.ylabel(
        "Mean pulse area per group (V us)"
    )

    plt.title(
        "Group-mean pulse area and PI setpoint"
    )

    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


if not control_history.empty:
    plt.figure(
        figsize=(11, 5)
    )

    plt.scatter(
        control_history[
            "controller_time_s"
        ],
        control_history[
            "dac_high_after_v"
        ],
        s=20,
    )

    plt.xlabel(
        "Time since experiment start (s)"
    )

    plt.ylabel(
        "Red Pitaya DAC high level (V)"
    )

    plt.title(
        "Applied DAC correction"
    )

    plt.grid(True)
    plt.tight_layout()
    plt.show()


print("\nClosed-loop run finished.")
print("Manual stop:", manual_stop_requested)
print("Completed captures:", len(captured_waveforms))
print(
    "Valid individual pulses:",
    int(
        pulse_results["valid"].sum()
    ),
)
print(
    "Final DAC high level:",
    current_dac_high_voltage_v,
    "V",
)
print(
    "Active target mean area:",
    active_target_area_v_us,
    "V us",
)
print(
    "Raw waveforms:",
    RAW_WAVEFORMS_PATH.resolve(),
)
print(
    "Pulse results:",
    PULSE_RESULTS_PATH.resolve(),
)
print(
    "Group summaries:",
    CAPTURE_PULSE_SUMMARY_PATH.resolve(),
)
print(
    "Control history:",
    CONTROL_HISTORY_PATH.resolve(),
)

display(
    capture_pulse_summary
)

display(
    control_history
)

display_columns = [
    "capture_index",
    "pulse_index_in_capture",
    "time_from_first_pulse_s",
    "pulse_time_in_analysis_window_s",
    "valid",
    "reason",
    "pulse_area_v_us",
    "baseline_pre_v",
    "baseline_post_v",
    "threshold_v",
    "pulse_height_v",
    "peak_height_v",
    "pulse_duration_s",
    "area_error_vs",
    "normalized_area_error",
]

display(
    pulse_results[
        display_columns
    ]
)


# 5. For Red Pitaya — stop and close

Run this cell after the closed-loop experiment has completed or been
manually stopped.


In [ ]:
# 5. For Red Pitaya — stop output and close the connection

try:
    send_redpitaya(
        "OUTPUT"
        + str(REDPITAYA_OUTPUT_CHANNEL)
        + ":STATE OFF"
    )

    print(
        "OUT"
        + str(REDPITAYA_OUTPUT_CHANNEL)
        + " stopped."
    )

finally:
    rp.close()

    print(
        "Red Pitaya SCPI connection closed."
    )
